# Causal self-attention

This notebook follows the attention implementation in `src/model/attention.py`. It inspects the score matrix, applies a causal mask, and checks that future tokens cannot change earlier outputs.

In [1]:
import math
import sys
from pathlib import Path

import torch

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src" / "model").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError("Could not find the repository root")
sys.path.insert(0, str(project_root))

torch.manual_seed(7)

## Inspect scaled dot-product attention

The score tensor compares each query position with every key position.

In [2]:
batch, time, head_dim = 1, 4, 8
q = torch.randn(batch, time, head_dim)
k = torch.randn(batch, time, head_dim)
v = torch.randn(batch, time, head_dim)

scores = q @ k.transpose(-2, -1) / math.sqrt(head_dim)
future = torch.triu(torch.ones(time, time, dtype=torch.bool), diagonal=1)
masked_scores = scores.masked_fill(future, float("-inf"))
weights = torch.softmax(masked_scores, dim=-1)
output = weights @ v

print("scores:", scores.shape)
print("weights:", weights[0])
print("output:", output.shape)

scores: torch.Size([1, 4, 4])
weights: tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.6114, 0.3886, 0.0000, 0.0000],
        [0.0876, 0.8491, 0.0633, 0.0000],
        [0.0836, 0.0465, 0.7973, 0.0725]])
output: torch.Size([1, 4, 8])


Every row of the weight matrix sums to one, and entries above the diagonal are zero.

In [3]:
assert torch.allclose(weights.sum(dim=-1), torch.ones(batch, time))
assert torch.all(weights.masked_select(future.unsqueeze(0)) == 0)
print("causal mask checks passed")

causal mask checks passed


## Use the repository implementation

The reusable layer handles projection, head splitting, masking, head merging, and the output projection.

In [4]:
from src.model.attention import CausalSelfAttention

attention = CausalSelfAttention(dim=32, n_heads=4, max_seq_len=16)
x = torch.randn(2, 8, 32)
y = attention(x)
print("input:", x.shape)
print("output:", y.shape)

input: torch.Size([2, 8, 32])
output: torch.Size([2, 8, 32])


In [5]:
prefix = torch.randn(1, 4, 32)
first = attention(torch.cat([prefix, torch.zeros(1, 2, 32)], dim=1))
changed = torch.cat([prefix, torch.randn(1, 2, 32)], dim=1)
second = attention(changed)

torch.testing.assert_close(first[:, :4], second[:, :4])
print("future-token isolation check passed")

future-token isolation check passed
